# 03 · Train Model B — the surrogate panel

Deliberately *not* hardened. Trained on the non-adversarial split only, so it
behaves like a typical third-party detector — which is the whole point.

The humanizer optimises against this model plus the perplexity ratio and the
stylometric features. Optimising against a panel rather than a single
classifier is what makes the rewrite generalise: tuned against one model it
learns that model's quirks; tuned against a diverse panel it learns the
properties third-party detectors genuinely share (PRD 8.1).

Same architecture, same loss, different data. **Needs GPU**, 4–8 hours.


In [ ]:
# Setup. Run once per session — everything below depends on it.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow google-api-python-client google-auth

import sys, os
from pathlib import Path

# Must be set before torch is imported anywhere — PyTorch reads it when CUDA
# first initialises. Reduces the fragmentation that turns "enough memory" into
# an out-of-memory error hours into a run.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Change this if you forked the repo. Public repo => no token needed.
GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"

# Either attached as a Kaggle Dataset named `ai-detector-repo`, or cloned.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

# parents=True so this also works off-Kaggle (Colab, a local box) after
# pointing WORK somewhere that exists.
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("work:", WORK)
assert (REPO / "training" / "lib").is_dir(), "repo not found — check GIT_URL"


In [ ]:
# ---------------------------------------------------------------------------
# Run configuration. Read this cell before starting anything else.
# ---------------------------------------------------------------------------
#
# Kaggle gives 30 GPU-hours a week. A full run is 8-16 of them, so you cannot
# afford to discover a bug at hour six. Leave SMOKE_TEST = True for the first
# pass: it runs the entire pipeline end to end in well under an hour on a
# tiny sample. If stage 4 completes, the chain works. Then set it False and
# run for real.

SMOKE_TEST = True

if SMOKE_TEST:
    SAMPLE_ROWS = 5_000     # rows per dataset
    EPOCHS = 1
else:
    SAMPLE_ROWS = 400_000   # PRD 12.2
    EPOCHS = 3

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}: "
      f"{SAMPLE_ROWS:,} rows/dataset, {EPOCHS} epoch(s)")
if not SMOKE_TEST:
    print("Expect ~1-2 h for stage 1, then 4-8 h per model. Use "
          "Save Version -> Save & Run All so a browser disconnect cannot kill it.")


In [ ]:
# ---------------------------------------------------------------------------
# Where work survives the session ending. Configure this before running.
# ---------------------------------------------------------------------------
#
# Kaggle wipes /kaggle/working when a session ends or times out. Without a
# remote store, a run that dies at hour eight leaves nothing behind. With one,
# the dataset and the newest checkpoint are pulled back automatically and
# training continues from the step it reached.
#
# Fill in ONE of the two. Leaving both blank runs local-only, which is fine
# for a smoke test and a bad idea for a real run.

# --- Option A: Google Drive (best for checkpoints) -------------------------
# Uploads replace the file in place, so syncing a 2 GB checkpoint forty times
# costs 2 GB rather than 80 GB.
#
# One-time setup:
#   1. console.cloud.google.com -> new project -> enable the Drive API
#   2. Create a service account, then create a JSON key for it
#   3. In Drive, make a folder and share it with the service account's
#      client_email (from the JSON) as Editor. A service account has its own
#      Drive with zero quota, so it can only write into a folder you shared
#      with it — this is the step people miss.
#   4. Folder id is the last part of drive.google.com/drive/folders/<THIS>
#   5. Upload the JSON to Kaggle as a PRIVATE dataset and point at it below
DRIVE_FOLDER_ID = ""   # e.g. "1AbC..."
DRIVE_KEY_PATH  = ""   # e.g. "/kaggle/input/gdrive-key/service_account.json"

# --- Option B: Hugging Face Hub (easiest) ----------------------------------
# Only needs a token: Add-ons -> Secrets -> add one named HF_TOKEN.
# The Hub is git-backed and keeps every revision, so repeated multi-gigabyte
# checkpoint pushes accumulate history. Raise remote_sync_minutes if you use
# this for checkpoints.
CHECKPOINT_REPO = ""   # e.g. "ByteCraft-9/ai-detector-checkpoints"

import os

if DRIVE_FOLDER_ID and DRIVE_KEY_PATH:
    os.environ["DRIVE_FOLDER_ID"] = DRIVE_FOLDER_ID
    os.environ["DRIVE_SERVICE_ACCOUNT_JSON"] = DRIVE_KEY_PATH
elif CHECKPOINT_REPO:
    os.environ["CHECKPOINT_REPO"] = CHECKPOINT_REPO
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        print(f"Could not read the HF_TOKEN secret: {exc}")

from lib.store import build_store
STORE = build_store()

if STORE.__class__.__name__ == "NullStore":
    print("\nNothing will survive this session ending. Fine for a smoke test;"
          "\nconfigure a store above before starting a real run.")


In [ ]:
import pandas as pd
from lib.data import FEATURE_NAMES
from lib.train import TrainConfig, train

frame_b = pd.read_parquet(DATA / "train_b.parquet")

# Hold out by *domain*, not at random. A random split lets the model memorise
# a generator's quirks and score well on rows from the same generator, which
# is precisely the overfitting RAID exposed (E3: fine-tuned RoBERTa-Large
# averaged 56.7%).
holdout_b = sorted(frame_b["domain"].unique())[-2:]
validation_b = frame_b[frame_b["domain"].isin(holdout_b)]
training_b = frame_b[~frame_b["domain"].isin(holdout_b)]
print(f"train {len(training_b):,} · validate {len(validation_b):,} "
      f"on {holdout_b}")

config_b = TrainConfig(
    backbone="microsoft/deberta-v3-base",
    max_length=768,
    batch_size=16,
    accumulation_steps=2,
    learning_rate=2e-5,
    epochs=EPOCHS,
    fp16=True,
)

# Checkpoints land in WORK/model_b every 500 steps, and every
# remote_sync_minutes the newest one is pushed to STORE. On startup this looks
# for a local checkpoint, then a remote one, and fast-forwards to the step it
# reached — so if the session dies, run the notebook from the top again and
# training picks up where it stopped.
model_b, report_b = train(
    training_b, validation_b, list(FEATURE_NAMES),
    output_dir=WORK / "model_b",
    config=config_b,
    store=STORE,
)


In [ ]:
# Sanity check on what B is *for*. It should be clearly weaker than A on
# humanized text — that gap is the product's honesty margin, and the two
# numbers the UI shows are exactly this difference made visible.
final_b = report_b["final"]
print(f"Model B  AUROC {final_b['auroc']:.4f}  ·  TPR@1%FPR {final_b['tpr_at_1_fpr']:.4f}")
try:
    print(f"Model A  AUROC {report_a['final']['auroc']:.4f}  ·  "
          f"TPR@1%FPR {report_a['final']['tpr_at_1_fpr']:.4f}")
except NameError:
    print("(Model A not in this session — compare against stage 2's output.)")
print("\nIf B matches A on adversarial text it is not a surrogate for anything.")
print("Stage 5 measures that gap directly.")
